# Backing up run artifacts / 산출물 백업

Collects everything needed to (a) redraw the figures later and (b) rebuild the
paper tables, then packs it into a `tar.gz`. **MIMIC raw files (`mimic_data/`)
and the derived parquet cache (`cache/`) are never included** — they cannot be
redistributed under the PhysioNet data use agreement, and the cache is
deterministically regenerated from the raw files anyway. The last cell reads the
archive back and verifies that nothing excluded slipped in.

---

# 산출물 백업

학습이 끝난 뒤 결과를 회수한다. 목적은 두 가지다.

1. **나중에 그림만 다시 그릴 수 있게** 하는 것 — 그림 셀이 읽는 배열과 CSV를 전부 담는다.
2. 원고 빌더가 읽는 표 CSV를 담는 것.

담은 뒤에는 노트북을 열어 **Setup·상수·progress 3개 셀만 실행**하면
`[검정 전용]`·`[요약 전용]`·`[그림 전용]` 셀이 그대로 동작한다. 재학습이 필요 없다.

## 담지 않는 것

**MIMIC 원자료(`mimic_data/`)와 파생 parquet 캐시(`cache/`)는 담지 않는다.**
PhysioNet 데이터 사용 동의상 재배포할 수 없다. 캐시는 원자료에서 결정적으로
재생성되므로 백업할 이유도 없다. 아래 검증 셀이 혼입 여부를 직접 확인한다.

In [21]:
import os, tarfile, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
TABLE_DIR = ROOT / "tables"
FIGURE_DIR = ROOT / "figures"
ART_DIR = ROOT / "artifacts"

# 담는 것: 집계 수치(tables), 그림(figures), 그림 재생성에 필요한 배열(artifacts),
#          실행이 끝난 노트북.
INCLUDE = ["tables", "figures", "artifacts"]
INCLUDE_FILES = ["cdld_los_paper.ipynb"]

# 담지 않는 것: MIMIC 원자료와 파생 parquet 캐시.
# PhysioNet 데이터 사용 동의상 재배포할 수 없다. 캐시는 원자료에서 결정적으로
# 재생성되므로 백업할 이유도 없다.
EXCLUDE = ["mimic_data", "cache"]

STAMP = time.strftime("%Y%m%d_%H%M")
OUT = ROOT / f"cdld_los_backup_{STAMP}.tar.gz"
print(f"작업 폴더 {ROOT}")
print(f"백업 대상 {INCLUDE + INCLUDE_FILES}")
print(f"제외      {EXCLUDE}  <- MIMIC 재배포 금지")

작업 폴더 /workspace
백업 대상 ['tables', 'figures', 'artifacts', 'cdld_los_paper.ipynb']
제외      ['mimic_data', 'cache']  <- MIMIC 재배포 금지


## 1. Can the figures and tables still be rebuilt? / 그림·표 재생성 가능 여부 확인

In [22]:
# ---------------- 나중에 그림만 다시 그릴 수 있는가 ----------------
# 그림 셀이 읽는 파일이 전부 있는지 먼저 확인한다. 하나라도 빠지면 백업을
# 복원해도 그림을 다시 만들 수 없다.
NEEDED = {
    "fig_pred_vs_actual":        ["artifacts/p_test_true_days.npy",
                                  "artifacts/p_test_pred_days.npy"],
    "fig_encoding_physician":    ["tables/p_main.csv"],
    "fig_records_vs_contribution": ["tables/p_stratified.csv"],
    "fig_fairness":              ["tables/p_fairness.csv"],
}
# 원고 빌더가 읽는 파일
NEEDED["원고 표"] = [f"tables/p_table{s}.csv" for s in
                     ["2", "3", "4", "5", "6", "A1", "S1"]] + \
                    ["tables/p_tests.csv", "tables/p_aux.csv"]

missing = []
for fig, files in NEEDED.items():
    for f in files:
        ok = (ROOT / f).exists()
        if not ok: missing.append((fig, f))
        print(f"  {'OK ' if ok else '없음'}  {fig:28s} <- {f}")
if missing:
    print(f"\n⚠️ 누락 {len(missing)}건 — 해당 실험 블록을 먼저 실행해야 한다.")
else:
    print("\n그림·원고 표 재생성에 필요한 파일이 전부 있다.")

  OK   fig_pred_vs_actual           <- artifacts/p_test_true_days.npy
  OK   fig_pred_vs_actual           <- artifacts/p_test_pred_days.npy
  OK   fig_encoding_physician       <- tables/p_main.csv
  OK   fig_records_vs_contribution  <- tables/p_stratified.csv
  OK   fig_fairness                 <- tables/p_fairness.csv
  OK   원고 표                         <- tables/p_table2.csv
  OK   원고 표                         <- tables/p_table3.csv
  OK   원고 표                         <- tables/p_table4.csv
  OK   원고 표                         <- tables/p_table5.csv
  OK   원고 표                         <- tables/p_table6.csv
  OK   원고 표                         <- tables/p_tableA1.csv
  OK   원고 표                         <- tables/p_tableS1.csv
  OK   원고 표                         <- tables/p_tests.csv
  OK   원고 표                         <- tables/p_aux.csv

그림·원고 표 재생성에 필요한 파일이 전부 있다.


## 2. What goes in / 담길 내용

In [23]:
# ---------------- 담길 내용 미리보기 ----------------
def walk(p):
    for r, _, fs in os.walk(p):
        for f in fs:
            yield Path(r) / f

rows = []
for d in INCLUDE:
    if not (ROOT / d).exists():
        print(f"  (없음) {d}"); continue
    fs = list(walk(ROOT / d))
    rows.append({"항목": d, "파일 수": len(fs),
                 "크기(MB)": round(sum(f.stat().st_size for f in fs) / 1e6, 2)})
for f in INCLUDE_FILES:
    if (ROOT / f).exists():
        rows.append({"항목": f, "파일 수": 1,
                     "크기(MB)": round((ROOT / f).stat().st_size / 1e6, 2)})
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print(f"\n합계 {inv['크기(MB)'].sum():.1f} MB")

# 제외 대상이 실수로 섞이지 않는지 확인
for d in EXCLUDE:
    if (ROOT / d).exists():
        n = len(list(walk(ROOT / d)))
        print(f"  제외 확인: {d} ({n}개 파일) — 백업에 담지 않는다")

       항목  파일 수  크기(MB)
   tables    57    0.16
  figures    74    8.41
artifacts    23   52.52

합계 61.1 MB
  제외 확인: mimic_data (4개 파일) — 백업에 담지 않는다
  제외 확인: cache (3개 파일) — 백업에 담지 않는다


## 3. Pack / 묶기

In [24]:
# ---------------- tar.gz 생성 ----------------
def is_excluded(path):
    parts = Path(path).parts
    return any(e in parts for e in EXCLUDE)

added = 0
with tarfile.open(OUT, "w:gz") as tar:
    for d in INCLUDE:
        if not (ROOT / d).exists(): continue
        for f in walk(ROOT / d):
            rel = f.relative_to(ROOT)
            if is_excluded(rel): continue
            tar.add(f, arcname=str(rel)); added += 1
    for f in INCLUDE_FILES:
        if (ROOT / f).exists():
            tar.add(ROOT / f, arcname=f); added += 1

size_mb = OUT.stat().st_size / 1e6
sha = hashlib.sha256(OUT.read_bytes()).hexdigest()[:16]
print(f"{OUT.name}  {size_mb:.1f} MB  ({added}개 파일)")
print(f"sha256(앞 16자리) {sha}")

cdld_los_backup_20260731_0442.tar.gz  46.5 MB  (154개 파일)
sha256(앞 16자리) 68c2e300af2dcf18


## 4. Verify / 검증

In [25]:
# ---------------- 검증 — 담긴 목록을 직접 읽어 확인 ----------------
with tarfile.open(OUT, "r:gz") as tar:
    names = tar.getnames()

leak = [n for n in names if any(e in Path(n).parts for e in EXCLUDE)]
print(f"담긴 파일 {len(names)}개")
print(f"제외 대상 혼입: {len(leak)}건" + ("  ⚠️" if leak else "  (정상)"))

missing_in_tar = [f for files in NEEDED.values() for f in files
                  if (ROOT / f).exists() and f not in names]
print(f"필요 파일 중 누락: {len(missing_in_tar)}건"
      + (f"  {missing_in_tar}" if missing_in_tar else "  (정상)"))

for pre in ["tables/", "figures/", "artifacts/"]:
    print(f"  {pre:12s} {sum(1 for n in names if n.startswith(pre))}개")

print("\n복원:  tar -xzf " + OUT.name)
print("복원 후 노트북을 열어 Setup·상수·progress 3개 셀만 실행하면")
print("[검정 전용]·[요약 전용]·[그림 전용] 셀이 그대로 동작한다 (재학습 불필요).")

담긴 파일 154개
제외 대상 혼입: 0건  (정상)
필요 파일 중 누락: 0건  (정상)
  tables/      57개
  figures/     74개
  artifacts/   23개

복원:  tar -xzf cdld_los_backup_20260731_0442.tar.gz
복원 후 노트북을 열어 Setup·상수·progress 3개 셀만 실행하면
[검정 전용]·[요약 전용]·[그림 전용] 셀이 그대로 동작한다 (재학습 불필요).
